In [1]:
%pip install Sastrawi

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
#1. Import library
import pandas as pd
import re
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

factory = StemmerFactory()
stemmer = factory.create_stemmer()

print(stemmer.stem("mengerjakan"))

kerja


In [16]:
import pandas as pd

#2. Load Dataset
data = pd.read_csv("dataset.csv")
text_column = "content_article"

# Cek apakah dataset berhasil dimuat
print(data.head())
print("Kolom teks yang digunakan:", text_column)


  date_publication media_source  \
0       2026-01-01       Kontan   
1       2026-01-01       Kontan   
2       2026-01-01       Kontan   
3       2026-01-01       Kontan   
4       2026-01-01       Kontan   

                                       title_article  \
0  Harga Emas Antam Hari Ini (28/3) Naik Rp 27.00...   
1  Kejagung Tetapkan Samin Tan Tersangka, Ini Kas...   
2  Asing Net Sell Rp 20,38 Triliun, Cek Saham yan...   
3                          Berburu Emas di Pagi Buta   
4  Hampir Rampung, ESDM Restui RKAB Batubara 580 ...   

                                     content_article  \
0  Reporter: Noverius Laoli | Editor: Noverius La...   
1  Reporter: kompas.com | Editor: Noverius Laoli ...   
2  Reporter: Noverius Laoli | Editor: Noverius La...   
3                      Format Non-Text/Gagal Ekstrak   
4  Reporter: Sabrina Rhamadanty | Editor: Ignatia...   

                                         url_article sentiment_label  \
0  https://investasi.kontan.co.id/news/harg

In [ ]:
#3.Case Folding
def case_folding(text):
    return str(text).lower()

# Contoh
text = "Saya Belajar NLP"
print(case_folding(text))
# Output: saya belajar nlp

saya belajar nlp


In [ ]:
#4. Menghapus URL
def remove_url(text):
    return re.sub(r"http\S+|www\S+", "", text)

# Contoh
text = "Kunjungi https://example.com sekarang"
print(remove_url(text))
# Output: Kunjungi  sekarang

Kunjungi  sekarang


In [ ]:
#5. Menghapus Mention
def remove_mention(text):
    return re.sub(r"@\w+", "", text)

# Contoh
text = "@user1 produk ini bagus"
print(remove_mention(text))

 produk ini bagus


In [ ]:
#6. Menghapus hashtag
def remove_hashtag(text):
    return re.sub(r"#\w+", "", text)

In [ ]:
#7. Menghapus Angka
def remove_number(text):
    return re.sub(r"\d+", "", text)

In [ ]:
#8. Menghapus Punctuation
def remove_punctuation(text):
    return text.translate(
        str.maketrans("", "", string.punctuation)
    )

In [ ]:
#9. Membersihkan whitespace
def clean_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()

In [19]:
#10. Fungsi Cleaning Lengkap
def clean_text(text):
    text = str(text)
    text  = text.lower()
    text = remove_url(text)
    text = remove_mention(text)
    text = remove_hashtag(text)
    text = remove_number(text)
    text = remove_punctuation(text)
    text = clean_whitespace(text)
    return text

#menerapkan ke dataset
data["clean_text"] = data[text_column].apply(clean_text)
data[[text_column, "clean_text"]].head()

,content_article,clean_text
0,reporter noverius laoli editor noverius laoli ...,reporter noverius laoli editor noverius laoli ...
1,reporter kompascom editor noverius laoli konta...,reporter kompascom editor noverius laoli konta...
2,reporter noverius laoli editor noverius laoli ...,reporter noverius laoli editor noverius laoli ...
3,format nontextgagal ekstrak,format nontextgagal ekstrak
4,reporter sabrina rhamadanty editor ignatia mar...,reporter sabrina rhamadanty editor ignatia mar...


In [20]:
#11. Stopword Removal
stop_words = set(stopwords.words("indonesian"))

def remove_stopwords(tokens):
    return [
        token for token in tokens
        if token not in stop_words
    ]

# Contoh
tokens = ["saya", "sedang", "belajar", "nlp", "yang", "menarik"]
print(remove_stopwords(tokens))

['belajar', 'nlp', 'menarik']


In [21]:
#12. Stemming dengan Sastrawi
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stem_tokens(tokens):
    return [stemmer.stem(token) for token in tokens]

# Contoh
text = "Mahasiswa sedang mempelajari pemrograman."
print(stemmer.stem(text))

mahasiswa sedang ajar pemrograman


In [26]:
#13. Pipeline Preprocessing Lengkap
def preprocessing_pipeline(text):
    # 1. Cleaning
    text = clean_text(text)

    # 2. Tokenization
    tokens = word_tokenize(text)

    # 3. Stopword removal
    tokens = remove_stopwords(tokens)

    # 4. Stemming
    tokens = stem_tokens(tokens)

    return tokens

# Menerapkan ke dataset
data["processed_tokens"] = data[text_column].apply(preprocessing_pipeline)

In [27]:
#14. Menggabungkan Token Kembali
data["processed_text"] = data["processed_tokens"].apply(
    lambda tokens: " ".join(tokens)
)

In [28]:
#15. Perbandingan Raw vs Processed
for i in range(5):
    print("RAW:")
    print(data.iloc[i][text_column])

    print("\nPROCESSED:")
    print(data.iloc[i]["processed_text"])

    print("=" * 80)

RAW:
reporter noverius laoli editor noverius laoli kontancoid jakarta harga emas batangan bersertifikat antam keluaran logam mulia pt aneka tambang tbk antm naik pada sabtu mengutip situs logam mulia harga pecahan satu gram emas antam berada di rp harga emas antam itu naik dibandingkan dengan harga pada jumat yang berada di level rp per gram sementara harga buyback emas antam berada di level rp per gram harga tersebut naik rp jika dibandingkan dengan harga buyback pada jumat yang juga ada di rp per gram berikut harga emas batangan antam dalam pecahan lainnya per sabtu dan belum termasuk pajak harga emas gram rp harga emas gram rp harga emas gram rp harga emas gram rp harga emas gram rp harga emas gram rp harga emas gram rp harga emas gram rp harga emas gram rp harga emas gram rp logam mulia antam menjual emas dan perak batangan dalam beberapa ukuran berat misalnya gram gram dan gram biasanya harga per gram emas antam akan berbeda tergantung berat batangnya perbedaan ini terjadi karena 

In [35]:
#16. Menyimpan Data Hasil Preprocessing
output_columns = [text_column, "clean_text", "processed_text"]

data[output_columns].to_csv("dataset_processed.csv", index=False)

print("Data berhasil disimpan.")

Data berhasil disimpan.


In [36]:
# Membuat kolom hasil cleaning
data["clean_text"] = data[text_column].apply(clean_text)

# Cek hasil
data[[text_column, "clean_text"]].head(10)

,content_article,clean_text
0,reporter noverius laoli editor noverius laoli ...,reporter noverius laoli editor noverius laoli ...
1,reporter kompascom editor noverius laoli konta...,reporter kompascom editor noverius laoli konta...
2,reporter noverius laoli editor noverius laoli ...,reporter noverius laoli editor noverius laoli ...
3,format nontextgagal ekstrak,format nontextgagal ekstrak
4,reporter sabrina rhamadanty editor ignatia mar...,reporter sabrina rhamadanty editor ignatia mar...
5,reporter kompascom editor noverius laoli konta...,reporter kompascom editor noverius laoli konta...
6,kontancoid yuk lihat harga emas hari ini di pe...,kontancoid yuk lihat harga emas hari ini di pe...
7,reporter dimas andi editor tri sulistiowati ko...,reporter dimas andi editor tri sulistiowati ko...
8,kontancoid yuk lihat harga emas antam logam mu...,kontancoid yuk lihat harga emas antam logam mu...
9,sumber reuters editor khomarul hidayat kontanc...,sumber reuters editor khomarul hidayat kontanc...
